# Vector databases

Can a library of papers on AI and work help us find evidence about worker productivity? In this offline example you build a small vector database, a collection that represents passages as numerical coordinates so you can search for related ideas. You then see how a reading decision changes what the library can answer.

**Run the example.** Open this file in Jupyter or VS Code, select a Python 3 kernel and choose **Run All**. Everything for the default run is in this file. A temporary SQLite database holds the results and is removed by the final code cell.

**What runs.** Python splits short notes on two public papers into passages, indexes their saved SPECTER2 embeddings and calculates the closest passage in each paper. To illustrate discovery, the notebook replays the ten highest-ranked papers from an earlier feed search. You can change how much weight each score receives, choose whether to add a third paper and run the search again. PDF conversion and feed collection happened before this example. The optional model download and Chroma example are switched off.

## A first nearest-neighbor search

What country stands in the same relation to Rome that Germany does to Berlin? The next two cells use invented coordinates to draw that relationship. Subtract Berlin’s vector from Germany’s, add the difference to Rome’s, then find the country closest to the result. The six numbered steps that follow apply the same idea of proximity to economics papers. [Word analogies in learned embeddings](https://aclanthology.org/N13-1090/)

In [ ]:
from math import dist

# Teaching coordinates in an invented two-dimensional embedding space.
points = {
    "Berlin": (1, 3), "Germany": (2, 1), "Rome": (5, 4),
    "Italy": (6.1, 2.1), "Vienna": (3.15, 3.1), "Austria": (4.15, 1.1),
}
countries = ["Germany", "Italy", "Austria"]
offset = tuple(g - b for g, b in zip(points["Germany"], points["Berlin"]))
target = tuple(r + d for r, d in zip(points["Rome"], offset))
nearest = min(countries, key=lambda name: dist(target, points[name]))
print("Target:", target, "| nearest country:", nearest, "| distance:", round(dist(target, points[nearest]), 2))

In [ ]:
def draw_analogy(points, target):
    # Coordinates are mapped to the display only here. Distance is measured above.
    def xy(point):
        return 20 + 68 * point[0], 177 - 33 * point[1]
    def segment(start, end, color):
        a, b = xy(start)
        c, d = xy(end)
        return f'<path d="M{a},{b} L{c},{d}" stroke="{color}" stroke-width="2.5" fill="none" marker-end="url(#tip)"/>'

    svg = ['<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 575 195" role="img" aria-label="Berlin to Germany and Vienna to Austria give the same offset. Applying it at Rome locates Italy."><title>Capital-to-country offset in two dimensions</title>',
           '<defs><pattern id="lattice" width="34" height="33" patternUnits="userSpaceOnUse"><path d="M34 0H0V33" fill="none" stroke="#d4d6ca" stroke-width=".7"/></pattern><marker id="tip" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="4" markerHeight="4" orient="auto"><path d="M0 0L10 5L0 10Z" fill="#23665d"/></marker></defs>',
           '<rect width="575" height="195" fill="#f2f2e8"/><rect width="575" height="195" fill="url(#lattice)"/>',
           segment(points["Berlin"], points["Germany"], "#23665d"),
           segment(points["Vienna"], points["Austria"], "#23665d"),
           segment(points["Rome"], target, "#23665d")]
    for name, coord in points.items():
        x, y = xy(coord)
        color = "#895330" if name == "Italy" else "#274339" if name in ("Berlin","Germany","Rome","Austria","Vienna") else "#5c675f"
        dx, dy = (12, -5) if name == "Italy" else (8, -9)
        svg.append(f'<circle cx="{x}" cy="{y}" r="4.7" fill="{color}"/><text x="{x+dx}" y="{y+dy}" fill="{color}" font-family="Arial,sans-serif" font-size="16">{name}</text>')
    x, y = xy(target)
    svg.append(f'<circle cx="{x}" cy="{y}" r="7.6" fill="none" stroke="#975e30" stroke-width="1.8"/><text x="{x+10}" y="{y+21}" font-size="13" fill="#72492c" font-family="Arial,sans-serif">query point</text></svg>')
    return "".join(svg)

try:
    from IPython.display import SVG, display
    display(SVG(draw_analogy(points, target)))
except ImportError:
    print("The coordinates and nearest neighbor above give the same result in plain Python.")

## 1. Prepare sources

Choose the topic and a rule for inclusion. Here the starting collection covers AI’s effects on tasks, hiring and worker performance. The two starting papers receive stable IDs, versions, public links and independently written Markdown notes. A third note waits for a reading decision. For the research collection, a local Python script used [Marker](https://github.com/datalab-to/marker) to turn PDFs into Markdown. Preserve the citation and link figures and tables back to their positions in the paper. The short notes here are paraphrases, not quotations from the PDFs.

The richer research format provided fields for 100- and 500-word summaries, figure descriptions, tables and metadata. Fields can be empty. If you index a figure description, identify it as model-written and retain a page and figure reference. The small example below indexes the notes’ text.

In [ ]:
records = [
    {
        "id": "nber:w31161",
        "version": "2023 working paper",
        "title": "Generative AI at Work",
        "url": "https://www.nber.org/papers/w31161",
        "text": "A customer-support firm introduced an AI assistant. Agents using AI resolved more issues in each hour. The largest gains went to less experienced workers."
    },
    {
        "id": "nber:w28257",
        "version": "February 2022 revision",
        "title": "AI and Jobs: Evidence from Online Vacancies",
        "url": "https://www.nber.org/papers/w28257",
        "text": "US employers more exposed to AI reduced hiring for non-AI jobs. At the industry and occupation level, this study found no discernible link with wage or employment growth."
    },
    {
        "id": "arxiv:2512.03902",
        "version": "v1",
        "title": "Equalizer or amplifier? How AI may reshape human cognitive differences",
        "url": "https://arxiv.org/abs/2512.03902v1",
        "text": "AI may narrow or widen differences in cognitive performance. Bigoni and colleagues ask how schools should assess students if AI is allowed in coursework and exams."
    }
]
productivity_question = 'How does giving customer support workers an AI assistant affect their productivity?'
search_question = 'Which study discusses AI use in coursework and exams?'

In [ ]:
seed = records[0]
print(seed["title"], seed["version"])
print(seed["url"])

## 2. Split the text into chunks

A chunk is the piece of a paper that a search can bring back. Here we choose a maximum length in characters and repeat a little text at the start of the next chunk. A sentence about a result can then travel with both its setup and its interpretation.

The paper tracker used a width of 768 characters and an overlap of 50 characters. For the first note only, we shorten the width to 102 characters so both chunks fit on screen. Look for the sentence they share. Each chunk carries the original paper's ID and version.

In [ ]:
def split_into_chunks(text, width=768, overlap=50):
    """Return fixed-width character windows, repeating the overlap at each boundary."""
    if width <= 0 or not 0 <= overlap < width:
        raise ValueError("Choose a positive width and a smaller nonnegative overlap.")
    start = 0
    while start < len(text):
        yield text[start:start + width]
        if start + width >= len(text):
            break
        start += width - overlap

chunks = {}
for record in records:
    width = 102 if record["id"] == "nber:w31161" else 768
    key = record["id"], record["version"]
    chunks[key] = list(split_into_chunks(record["text"], width=width))

first_record = records[0]
example_chunks = chunks[first_record["id"], first_record["version"]]
for number, passage in enumerate(example_chunks, start=1):
    print(f"Chunk {number} ({len(passage)} characters)\n{passage}")
print("Shared text:", example_chunks[0][-50:])

## 3. Embed and index

An embedding gives each chunk a coordinate. Here the coordinate has 768 numbers from SPECTER2, a model trained on scientific papers. We use its *proximity adapter*, the version of the model for finding related texts. The research tracker stores these coordinates in Chroma alongside each passage's paper ID, version, number, text and source link.

The notebook includes the actual coordinates for its four public chunks and two questions. You can run the whole lesson with standard Python. It stores the coordinates in SQLite and calculates cosine similarity directly. An optional cell repeats the first search in Chroma. To try new text, install PyTorch (2.6 or later), adapters (1.2.0) and transformers (4.51.3), then turn on recomputation below. The first model download requires an internet connection.

In [ ]:
BASE_MODEL = "allenai/specter2_aug2023refresh_base"
BASE_REV = "084e9624d354a1cbc464ef6cc1e3646d236b95d9"
ADAPTER = "allenai/specter2_aug2023refresh"
ADAPTER_REV = "fb592d9375e91b996957c81824b30d32250c1a72"
encoder = f"{BASE_MODEL}@{BASE_REV}+{ADAPTER}@{ADAPTER_REV}"

def encode_texts(texts):
    import torch
    from adapters import AutoAdapterModel
    from huggingface_hub import snapshot_download
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REV)
    model = AutoAdapterModel.from_pretrained(BASE_MODEL, revision=BASE_REV)
    adapter_path = snapshot_download(ADAPTER, revision=ADAPTER_REV)
    model.load_adapter(adapter_path, set_active=True)
    model.eval()
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.inference_mode():
        first_token = model(**inputs).last_hidden_state[:, 0, :]
        return torch.nn.functional.normalize(first_token, dim=1).cpu().tolist()

In [ ]:
# Stored float32 coordinates from this exact encoder. Expand only to inspect the cache.
import base64
import struct
import zlib

cached_encoder = 'allenai/specter2_aug2023refresh_base@084e9624d354a1cbc464ef6cc1e3646d236b95d9+allenai/specter2_aug2023refresh@fb592d9375e91b996957c81824b30d32250c1a72'
packed_vectors = {
    'A customer-support firm introduced an AI assistant. Agents using AI resolved more issues in each hour.': (
        'eJwFwQkgFmYfAHAhSmMoRBav+yi+Usn7/J9EKlGKJioWmg5dpEOlJXKXuyRHkSMkOQrv839qjNYIRchRqdCQqWQlZd/vp5mf'
        'iplWy+iKP6qZmVYERqimQ5pvBGb3NQlX+9uBw8F2zLngCf2NifD68CCe6vUm+2T1qwaaJXDV9/sgkReMMqec6cOvh0VPzMow'
        'f5sjyKunEFGbMhdcX0VFCab09RZPGNrage7HOLgliKP2TFcwVAnC2ZGeZOfmvyDjjD7csn6DZYtus+z9n0Dm4g/c5uRZVJsv'
        'JnQxeS+8GRfCLvQowJbceF72/i9US1xCNRwbYMIoFoqfK9IazcW4LUSarguR5StkW2F72mLatLUL+jvPARXcw0tPfuSraRRE'
        'iUsLL+6JE5XU95PQw2ug0koGw+RCqNkQEMnZrjBGtFGw+iRRMoulF143wJOsX/j2XR0sOeQCbZtrQDu7f4dn3VEw6hEA6Vt7'
        '8bAgB10W2ULGw7/Zko9JUD+wlvc/CK8iK+zohnVXWepVSSr580t85mRKnJ9WQvhpc1qqZ4+Jw2kgMTMMtW6KgeD7ffJQtYvZ'
        'jZTBbeub8IA8RfU9hWQiJJX1vZlkT85so5o2iqjAPoHKVDZusZ4L1Y9yoN7DiMbW7uEZTW9x1ox0NBleINJRMOIT5lEW3aZW'
        '4DWowv/zOoQHwp3Q4IYfrtrkBDtfV4GKRgAJU02Gtv7fYKx/BT52vQuyTnt5q406335Rm9tySxA7+RaN46uZs+FVHP09iJfN'
        'KmIuxUZ8pdcmGs0S0T93qygroIj+1fMUSh8o8Q5rf3hyOhjtl0jyWk99LDiwEQy+rMZavTzMvlKNJ6wZHMtagLn/9OBL30Ty'
        '5p4r5HU7ooFKHtsg3g9J7Ra8oC6S6m4Q8N9XW3IdgQRtnywEFal/2OcYHQjvvsmWnxTyim255M+VYnTcXR+kr2tBSdhcPjJ/'
        'O18Um4JVHUHC3Ihx8r7IFIz32tDA837w4oEVtASOsBLHUoittUHjfR+Ytr+Ayfy6iWsHmVOf1Hjwsl5NA93m8J1pDbghMJQ1'
        '2Zdh5kx5mBbQihc+T6f2Wl0omTBJlPZdZeJ1Z6E9aCVaj78h+wbMsC0hGLWEP8LRKyaQFSfJH1zrItGz7WG10TvmpjyOAuvp'
        'VELGjrR/CsaFJUZ4siEUPhT6wYv+NHJLLhfEfzVn9dOG2PXniigzoxW9fjSkYic92HG/EZyVPAzvxOL5d99SfLGxHI56ysCX'
        'tYo8uIbAdXlDmtP8grW1e2Hq/jLmMEeFSrYUE2G+Mf2wu44tuvUQ56k4k8SN9yC4ltIL5Yk4uj+M7S8NhpTrvqRmXq/Qe64S'
        'xFkSekBnIWlRusiqUlegvH0STwsQgFGxM3j7nwT3Mie6tOliVY9WIg7tOcPEc235x/OH4PKkK03/XEXM9h1ES9sFfNoGVT5w'
        'o1v0KrAAxjoriK6XLj+SuIYTqkKKy8tFH49bYt7PKazmzhh8X9qCkfUeOC+rGNjtVLw7OQTycF4oLQU0TdaKTcybw1OrmqFt'
        'bzuZE2vHg8eU8eCTbkyuXwgyVAVD3vnDlNpuUDv4I8qdr4HgLaW4022caMk9hNqeVtCFajS2V0W3kffQuhTxo6s0FXTJ0rd/'
        'RmCm0JNt1mOYHP8v/u3YJhQkFGJV+whxUbiEh9YTWiBRALeWvyXHvmjTk+waKJW7Q4KsPvzSGY/DG93plRtR5OovVQS+x8MO'
        'ZV0W9qiJOdqrItlvhm9bXFCwsp81GVcIizquwOmVWbCHTwrVX33G2HITni3Zh/4+75hXvjkv0X3KPBbXsuO66nT9/T7mYa7O'
        'D8V5cUuBDL97fRtanYzCstQ3pGz3IKnjSpzaJ4ByTMTy17ozadHkKkq6v+JQQyM4uMeg87CIXV0Wgi0DvaL78QS9Z12FdC8D'
        'MJldDp8cZ2DojFqS2jwdd53Sw9vD+VDMx+HyXW/0MRhEKeuv2GFQCJN+rqBp40q+eG6ga78+JmIiNRpyZT8NDxCBy3MlMhwQ'
        'TWMCbuNRp0asjZCj+0wl6fn0S7D62wGIcNgh1Fd8AYczp/BrqDfctSjmxi5J6PeXND2qKcTJW1dIw0ZXzK5pJh7y8czV7zlg'
        'sTvPd69kgceX8vuSz+Cb9XE4+u4/4pU/n3YOncLdOxpJ4hFTy8yhw1jblwNPT5eyy6ae8GgiRFTAGrCv8RS2FmngbunTONQT'
        'Bc5Z91mpZbyo4KAm7dixGJPnt2HOuBZ/9liM/XYvBTzmq/BbHY7sZU4aOJiJ0F/nIjv1Uh0M45/DS9UV8M5tOvlUvwdGlcvw'
        'j1dyeMJYyPO5Mq3w7CO+bznZlGfPS5yiYc3dnfyZ1A/0UelLUmGuyFOtTLBl80ZIvj+Xj6V3iywGCuDZRUfSM9zBXpgvYmdM'
        'h1mpZjQu1Okn/i3dxKjBAY9J7sHI/hQY9DSGO3tTsLJ3H9TUeJPrp3yxw2AV3zBziGkUTsHgmTzIlM4kmwPdcJv6K7AriiYa'
        'Gu4ouVQL45VfoIV5r1BWoxwCTP/FyugIiFcTw/mf5fGniDVYMceXj7S2wKMpU7LZQopXik+j5SemcYWJV+jQPkkGBR9w3Ysa'
        '5mMjTfobtDErWI5/+d8oM3o1hF2lSZiukorjJQdp4BsXYtswhdEHDMFtbTlKtQ/hPA0Vau5sxA88cMCPBREW00qysd8jDnwO'
        'BYNGaCJeqTDg+aX/w58mqiFjfBkfKF2Ax6R0SJxaMsa0xoGbyz3Q04jAp+VbQNF3iiQbVsIup7k8Ry2c+6ZfhrBsbVpPNbE1'
        'YAkfPKRNJ/gU3Dy3CjOqlKrW94vTnp85+N9fQ+wi70CuiRX/1S4SHEut4JzlKsht1OGfGx9heLE09/qWD9D4iYX/Z0t36Mnw'
        '0F15ojGjELq0OFy0M/McdlwKQVfvXRBe3omZUmeJqpckTOhXg+IqAW12mUO/1DyGGuVLUOkTiWT3erzn3EUcFj3BWzGyTBle'
        'oPY8Ca7xH2O9m7aR2vKzqHvtC3v3PYm73khCwyN3yNqiZ0xCv5K98xUIC6bL8q0/iPEEe2ner/APixDVsZKymfDLIyn85NCJ'
        'XeMnYEmBFj84nCfq8+uHlvPaTLFoAGNGnfideV1EaGjHtrtdQ/mGKHx8WY3qjzWz971i+K2gDlOMDVDztDdfeD4V5kqr0QjL'
        'YVJvolLVMjBBhusjUKXxCRbKp5IbkmPsxNYUPPjcFp1mR2JWTgDPafsXX6kupX3ut1EZNLlOd5BwdsNivFLzBax8lpFved2w'
        'rKxIWCMU4721SmjdlQ+3QzOFsWsuMMWCU5h/LhYcIkcw4ICArN19Aos/pOCsloX82Ol77Ac5f5Lh/ydJUrjE1C2aoTqwGZce'
        'MePrtkvTmE4fNuyyHitPO+GK151gsW4ZpBTM4HNtBkCbzuOTo12goNuEcV2R+PqMEtW/rQxC4wx0eTzMdoi3Y8/UHXa46AFk'
        'vvwAnfsfgx9BLNxpiHpxa3nT20T03WLC/v32FTpzHxPUOIfyb6XBfaIejcd6yPI/BLSgxIbPycwgHjHR4OCRQpOzxXmF03b4'
        'lDab34w24na7LelILtAZKTLwTWULBD79Rhp8qmGG/nJ0mBaEoX47iEJdLzsY5ULNntvC5tHvoHB4Ib8Ro0drbfVYT9JeKti8'
        'kWb0RtDfW38TJdhK8uuXBknPHX167Iif8HVhILR0TuJLfzl6Jt0EJWfto3cixWi/QjzsCVKni6edZXX7WiAmw4xn0D4MHsiD'
        'urCfqM6oCq/Ok0e/5kwU9M3i4ZPKpMBqKd5olKKqf98C0QIzuGbpQWuDfiYJmtcsWjtlMKIsHtf8astnW8mhTJgal5F4gDsr'
        'njL14Xu4Zb0q/z8Sm+ra'
    ),
    'Agents using AI resolved more issues in each hour. The largest gains went to less experienced workers.': (
        'eJwFwQlADdgaAOCUNlo0LUqlhSKhiUL3/CdtVMptJRUxVOM1SkiJSZGIFipZSgtaFE37mO75TzIlkqSQUDIoMiVXWTN630el'
        'X+MHpyA6tPwVKVifAJ0xaVBf44Dq/kEgpRSJt8qAr2rqAdfqKnAXe2NIVo7VCLlbd+CCIi5LSQD57lEM0xHQ0t8vsQ3q0tzg'
        '4hIw9HUnBrYefCQiH4RNw5DkVwH+hpbc174cBkJKWUhWE1wo6UQvbXcwyGqCod36givWU7mvgOBY+Cc44FfIPrepYt/JYVHg'
        'matkzYp6GGv3A/Pvm7g44QwW92tQHc3LJDiwDp4HTqc/Mo0w/EMrdAYU4v5/NGn86ggqjOiFkOxikEzbhzX3pTnVFRPlAE9w'
        '2XAUE9+0iTK6Juo8bSezdxrp9O1/G0josAx+L9mKbsec4KG7L40Jug2C916848gbdr08iMoEA2y1kaYGL4ugfd5lEAbLc90i'
        'Se5cliQQjglx9LsOHG8g3KCqBGNPrqSFUptxa9oEXD2+HyPSauCn1r1QrWFK8yb9j/waZkCfKjti/NRG0vbnMDlVYIzvzkvQ'
        'jH9U4NGXRuJec4tY6E9DH9+3AvGfxnQ45DNbyxbQ7hcLuemAFHheSYbg12L4YhXOozXn8Yeq0cyap+GmgOu4YO4h0efJl9Bw'
        'QovPqJJjEc6m6Jtczm7+kgads1IhwECdTU46Ayl3E0lYfRo5evgmyJ3eyPcZT+dGlur8a2kkXI6U4U/Ge9jOawU4/iOey5sr'
        '49UTs7j0ntm0YB1Hv+sUx4sz6MgzDerdqs/f1J2GnQ+zsNrjGvqt6BREpaqD67TDpMQ4g4CJDFd91kReJ+li0cm5vDr/NxYq'
        'XMLsXyeht2Qu80vVo0b7TPmL07upow1Dm9xtvFdmAFrsG8CkKQd1I4+BZcd88nCSDrcZTBCA+CCJc2iA0JPxEK1nyv08KM/e'
        'rY/jV9dAnp0XzOs3Y72usrQ1ogHqvI3gm3gbhg1JocX+EeZjp4ckJ1u03SOenyldQids/WF84UcgudI8u8iL1awIQ5Jqj3LC'
        'pRja4MYWR4phvrwT7lQqBReXWYLrTXJYnN1DXN6XwLfPeZgxmo/5xAW1P4aDAp7BjMbVovIlDgIb82OoXWSKCUmGVCvOwiqh'
        'NQfTDb8Ixi0WEPPDfWRHpQlx0raAdVmhEOqQiSqiUnQ5NYSHDoTQRsNU3PFbOEo/64cO+9W888Z0fq/jNvC+alZ+S5pPNVYD'
        'p6gWctJ7L1bmlLAZzxuJfak7TZDaRPJdPanE8hFRp3ksTj23BbTHLkBj+ix6JCAK19nb4nqdHNLt0SvavlsOtrZ5kOh0F+ok'
        '0QI+CjtJ86N7zNu9kLu4vBOE7RkhmdOyoXtOJHXzPQTvnZegaFIBU5yQ4IM+60RFWir0aGwOy6q2wWwTO35iSMgNYp0h+Ik5'
        'GQl8BIdG1bia2gG+4I0E//pZlT2+poS/N6WSZVvtQGwoZm3Wa3HxcgpzPrmRlLpL0OP5g33bt4p2SPqTmHRlHuTTDl+r1oNV'
        'dQKX84zGIFkhapochGANDYwxzGYzzirR93Z9gtjL0SC9RYweY4bwwXYCJBMTIVxVgee4N7K7hQMwamnO37oF042zzoNZbxeq'
        'G9xgjn73MSgwAHPDdesS/Svxt4O5uH6oFaO2BNHXBvUC/UW+2FQFcHa+PS29MsKiG9Sprfg5Prai9PwnAWo06+IXaWWMX3yT'
        '7dqwGoUTn8mLNUISqqYs0N28BM1yisi2GBVaGlkAlXF5xGcog22Mm81LPpnwm//1sJI4Cf50URur2H+WvQn+F3asXMW6Fipy'
        'y0xbHhdhxMeuBQuy97zC7UeMQVDnCO27rXhXoxrkb3oB/oODsFHTl/Y0IKZduQOzrU+TiJ6f0fSb+OqNpE9EJ2cCvc7kQOAM'
        'M7Y/UE3Q3KWOdRYHWHtyI9rq7CV5XzcQd1tl+j/5ExhX+wMLTf7AUS1DWD4vkximV7HFafo0KnobLC2aTnPujkPdk2uw5+ws'
        'GFnrQxNyenAwS5JfDOkFw+S5VKE9DHJHe8kDvRzUO1YDHxev5HKbU+DSw2yetbkCZ1Vb0FupX9j+mFPQt+84VguOocGstdg7'
        'Jxce5v3En6TOwOIpZvxg9ivICToHw9P2Qtl8GarxhyS2pR5ikYGWy3MNhonu0wFYUfKGtHw8DCOWm/D0nEJMtDbCSu0yVhle'
        'i6+9fUF/wSp87lAJl9wvgvX3ajT+tQNDLM5g1IenosDaUbJWaSG/l9ZbF9NqBXdTt8MCpWqSFn4QlvVmQl/VPajbL2RrjoRC'
        '/bRG0eCd2fjlgwLv/leGnp+ZC6R+CvSel+b7YC4Z0U7ixvpLoTmtihz/Q8jLu5JBw/EO5N6T588T5uKVLg04oR0HoswlSEXP'
        'mNiwjimtEeGu8b/Aed4LYj18E1unBmJ/mAO5l18Nr88twtv60fhtMBgTJpvjzAljPnMgTfRZewIGUouh7VKilZfsBeblUAyb'
        '4pPBY7CMde9tYfPeNeKoRh6kdSxG9QRH/kJfCczc1NEtOAlBcRQXTbLjTgZatHlpv+iM3VNsbXWhvTqrcEB8AufHUPJK8Bb5'
        '3zXsX5dXAu9XHdi/VAoX3V6FNg+60PJbFzNT+gePtvtTI7+TuGi9Pq/atcbK0bsUr/bM5HaPTenSp5r8pbSWQF25n6hu20g8'
        'I+tJhZQCJE5Vw/5RO96OihC2To7EjljwVpMa9K24R07FW2FgTSHMvGhFLY6J8XufDHb/UIcApQLQ/isXh4P9uWeNIfisU6AO'
        '6cH4KmMpb898B1N3noXZS64hbpDAULEczVyTAt+vD5O2ynrYNceI39lXKIqR+pu01H4BYunOI7Y1YH6PMY+f0wlfb6Tj/b/s'
        '6eopmvx8WqHVFslAKme4DCbl9xHx3p/w3PowCHqSilW6IrI9fzWQ9nfgHpABbyYeQVXRr2A7X5Ga+1ijYlKtyPHye/J0wQhe'
        '3aUlaqx8hcnvG1B291fis24XdNocQf9kVVQ+G8Iv52rx25OC2fOL5qLItzvg9yOKpFZogFHCJHTyuYDTE+owoH2zYN/9NUBO'
        'LUGVXAHrL7pM/nt5EoO6VOE/u9fg+edeVGt+jinPDvBa0XUY67/Dsucuw44TClx+ykbqFpNNyhXXw4Pvruh7u1Nkt0OFpwyU'
        'wW+hvfDY+RZzUz4IX24qgaxEEF5/pojNyxaC3jcv9k+HNpf/pQO3LTqA85zj+NI7Y4In0qa06cAxJmU3h/uLNtYVKyhhT+9U'
        'mnl/NlQ9kqVDd8uYWWwu3m7OxldMCMs/ZzJ/N3UiXqvCGoouEW2VfNzsPg2aZ3SjneCL6Fc9hqF6g0xuuhqIT8tDWMVmotX9'
        'HZhgkH3zncSnRA/BQ9sLLMz1IMrJHsIrG1tAboY2vWz7GG9kS9GytdJcZuUNUEuURyZ0wBXPA2HmJ33iFDmZC7PvkZVXrqHN'
        'kSRRyp5K4uzzAFpk68GoQR2rxpvZ42JbfiskC4Otu9iR7jHgPclwsTIZ83XUIDp2K/psLYfRnSuof58Wz2ptYBWVXnD6z520'
        'rVOWBx/3g2P7L6AdKvCJW6ZUNKBDN0XlkaKfnDH1pBC4uQqdf1obX77U5Jus5QQyvyyFRG89mrwwloUPVIAEn8kD5D6B2SpN'
        '/PtnPZoSaUM/TnalW0JLifXqG7hrrgbGqNdCeYUrnjrXAaTPhJfLj8P7LCfmkmVAufkkekinCh7E3QKdtMPMTvU6RLSoc5XY'
        'K7jc1YuJzEdB+ut07i7jghvWrcKR25rczaGMDUk6slYnLfpubSosdxMTBUiictkFxCXqomiBhwS36tuHea6WvPZHOpmn1oIN'
        'nXfRR6UAz+19hL8oluD/AU8D1io='
    ),
    'US employers more exposed to AI reduced hiring for non-AI jobs. At the industry and occupation level, this study found no discernible link with wage or employment growth.': (
        'eJwFwQk4FVgbAGBLKjtZQyGKSosy4Z7vRBgKMbQXLcTDFGNGGRVFCyJLRtS1zZUKSZMS7vmOrURokakUaRMxll+NRkj/+9Z0'
        'qdIPGUY06ms7u+JnC94qXHClzBUSeLTYtHolahUs5EH92RC/vh/mp0ygo4oNHqs7WZH2pBBTF+9Bu9kaEL5RCP55BtjjWYHL'
        'oj3wxxkrsXfpPC6ZXoGWy1rJOeNHoGL4Hl/sU6YPb79jb/JNoD/cFQSOfzF/TQl6a30fLsJTuEEQJ17QoU6DVVvY6lWvUGmV'
        'M+vPcieOwX+Si1/GoAXGcHT9Kt7sEkZ/Pa8Nac0SNEj4GPrMz7Eg907I9+/CjGcu1LPYjbZtn017u04T70f/4get9bzq9nkU'
        'KdWwvtXrUGTWTqbuiyH3TAs2ay+kFz1kIOlVsvhWqhrUBwlBJukl1M3+DJ8tEni8xyd8uUmb+q0aBO5lAlsDpWH16WawIm+x'
        'prsTXT4qC8bVtWFmrRqNWZmG+iEG+BuWwVT/HNgDe+GlihAVDZdTlUN2JN9Xl0qNSsLHf+aBRu0eaL5uDs+t21ijlxfGjhWh'
        'o1cz+Cn9QMqCgll/WTk2Wd1gs3q3gI1RHvEMFIJdYDOWVr8S6HmtoJnxAvCd78K1tXR4z+FijEn4gLG64Xwy9CJZai8D2x2G'
        'MCPgNLMpkOKS4lMoWXaCTXh+Ebc+LGQGn42x6Vgt7HcwI5ofq6H0j4V8j3MKhvarcT07K1I9cB5Dh+JA+eYLzAhezy9dn8kf'
        'vtDnl4MC4cLRx2xnShzgVDy9FWdK0w7Xol+wLsj5v8Tm8w346o8AsqvaByqXS8D2+hFcpjuHd9dRVDjxhB2W1Ob/fbMC+ZAi'
        'ptz1Ar0iQonKyw00yl6Dk+Mb6NEL57BOsgTTsyfh4nJXGDaOwbVvDsBu5WFB0Z25fMGizWh8IBLs0qSpn78HvCkswdk5Bqwq'
        'MRuPqjZgw1g+Si7KRtlUObr+02u4v9aTJAwYsWJhH048yUTXni6scKzBtsYD/A+t6XR6eBiEtrrTgyZr+NvhMpz5UxmLlI5i'
        'Dq0fsLXimMDnSybYLPmCxTq/glBhLt8S5YMHA88KNn5eCjzfjWn4BLPR3haxm68s3SlW5GWKqVAdpgu2GyR53VJ5JHv/haBP'
        '3+FrpDHMTNelqda7oHXXOxKwxRDmzbwgiK10IMN/N2GT12ZcpvE/9tuhODrXZIfYwK6BZPynho0jlrxv/ArrKVmCndqxgs91'
        'q/jD5QhhTvnwNNcbKtW2gJ6PA8y3LoSBhBgsa18EhztVSZ0DF/zqqIxVWdFgKLKmUl9jcHegJNBnQXjsjbb4kkWd2C1XAZdm'
        'zaPHzRdTUeQEm3OqhbimmfCM0UCyKTOXhYgmWVlVCl3zzBJNM5NB5qols746DzttbNCow45qt6Wg+Joad//dhMtf68bkXRq0'
        'YUkJLNl5CZZey8Z857V80HQpV3pwBn4+k0n2eQbg7tUV0Jl4m/z+IplljVTgKv9zGLXgDVhPVaP3oD49HumDfvPN+K7256CQ'
        '2AJFBmf5Hd8hHPjiBjsmOyFCeifevbkSYvMGoLTmBFNvKxJLPJtAEVjAystKdLuoGJ6PviL4VAHWbV5I69ep89IWOSr4n5g8'
        'X1NZrqLAmHK8Mp8UleGSpEJx97Yq3Nwuw98uk2LccDqVmi9lldjVh0U6/4NAdKI3XrwWO48/BjNtEbs7JE2vrjBlw7VncVbf'
        'CMmxWkMzvYzFsYHucPpPE1JcJEN/Lt2LOzvc8IjqH3hVaIILe87g9bpUWORizMe9rmDRkAdysTfYBjjhfxFImmV16bsfwqF4'
        '/XTeFCvLrZQ8uXODNxq+VuZFWwrg3KdJwa/bp/CX4Rx44nxe/MVJlnqZdILCkRoibzoKn07HkvEMA/i6ejm+k0wlU9abcHNb'
        'Obl6ci6zL6+BkL50It9YydKNTPlU0w0oj1CGdQXydFD3I1tQVIaaRffw8hwRLElPYapH1PDQ915y4GsJ+BU3k3i9DEgxzRGb'
        'nwrGu4p29G1DGvpqunDXSQvoijKBsAhJ0KzdT4YsRJi8Mg0KC8L557enoO8nG67xTpkrTISBv8Y3IkpLZSEDUiBK1oBy/xj8'
        'R90Ll+9YxgfUhCjcM4J5T07DysgCCLmdSjMre1msanSlW2MEvt5obfPNjcD2vvtQpDQAeb4KtHvzM+sCew0u0s9n7ot/Y/b5'
        'N7BWX5FutJsFj253Q9y3QfCUmct3WK0jadMPkG3nqvCBXDsTQyc67nmK3ol/gbHdGrjnfQE+t8nQsXMbqZOyITyM/xG07uYR'
        '/7QUEpv6L1Sk2/NrMQ9hRLoSypOdWeJPM3jg2kIM7Szk7u/jQCzwggnbI9wxPQTe5JWQtSmT6LDuAa7b0o3/OqdDurMR3bFK'
        'DR63+YFmTqfYL+ETXB/+vaLAKFkcnFXOzHLHmK6mKkgeiYaLzzpw3a4O1tDxiZh2buUpvmLM0dGlTrc9yX0FSXBe2oS5owUw'
        'mLQYQhqNmeJUIIbvVeaqvyUJhs2OQ0naL/zsUBiZ2v1dIGOpSp5ay/ODzk3EfO0dsI82Ztf7HqKC9Grwz/1bIDLLgF6fYJJ/'
        'Q4n3Ge8nUvdzoOFAOWuusEVBxm7y9rQ1v+CphT073DHEcgYNuOyLErX1GHpNntZEi1Go04OLTX2AFnhyE/VOYh6RAx8ztmPB'
        '/WxwTvIh2TKv8bC5Di+a0oXNzSrwdECNe2Yr4fEJjko+s8COvQCrq/tg9LkDb5ioB7MZJ8Dy1iDs1VZETWcV3ieSheGBE3RG'
        'ZwFWpDCUrG6CO5oW9OrdKuwObcOsOZJ0m1Y9ZNZ0sHlHTOn+7ZQvmZ4E+aozaLt3Guiqi1DC2oa3vw9FV9kF9K9oMyKlOQrD'
        'snFI+72JulUUveShyx9E/4CihGncOaiRvc4ZwbAJZNn3hGD1ZCGk3rq7yqRXH4rvPGdmO91A/fo1XGG7FbXSsohduhBLPAWY'
        'vDYH5Xr0+A8/noFB83swe5ojH+88hPf1/HjpCj0sXaIPMa4ijFnmAIqBybjpZD2JTFTn5Z3f2PjxMXLZwB1d/c9CrHAe/uNL'
        'scf5PSgO5qC3yARHxqrgwvNB7L0iyWuu7+PXAm6D4a7vpOSxLuQKFvFHn/aBhFEEKkxqwPDBrfjo0C6cN+TEbyYY0CcLzKkw'
        'LwF9pM1gWD4IaYc1NL+5CU0lUmC5/ixGDHXj09w68ffMHeykiSNPblXkhvE1sH9TFc6/Y4Fbz92AhKZbYu8n50m9y1GQnpsL'
        'Tv7T+FDHSj4tkhCnvXbwoC4MSOso0xzZAW4WV8C+OoZt2qZJ93isQav2GJipJ8NHBhfA3bjDeHShAbxvHyHml2fT35+Xof5y'
        'TZ4l0U5SPraiwZ9yEP04B/XfKQpkzE7S04IhtCffSZHqSazIbUWDeklwO58Bg5pOcKZSWTAr1IMFb0CojfSCxsAqq69jQ6S/'
        '/iGk3Si1smdhOC6XybJlx/GE337y2vc4U33UAgWZRuxGWTzU/jwMoznOTFkrCrQ7CyDMXoRJiXLYLT8Nf9y/nKYH3sIvG2Uw'
        '+XgBCg9+RWZcBn+Hy9EAiQdwyjUbe1OSkLXWEv/ZSfj5P3/xfoczLChgLa0Ut0Oh0AIUDmpQsaUFv6CwmH5bPI4tHipU0mU+'
        'zbskBN3FV5mtbBMK46VIi20ujQu/i48774HLlwV8bq4mjf9rH2TsEcO9fXPps/Aasd2KCNj8IQvYFVvAXxDf9KQgrW0jq4dO'
        'QvHNW+JjZ0uxaYUYi0f2Yde2QrLQUgdDujdgc+PPbE7WFdCxyIaCsWIYccpC4l/GomaY4inYyI3Gt6FVRQ5GdnVhip4Omtze'
        'xhJjlPj/AVbf3lw='
    ),
    'AI may narrow or widen differences in cognitive performance. Bigoni and colleagues ask how schools should assess students if AI is allowed in coursework and exams.': (
        'eJwFwQc81vkfAHAiHqPElcwzIinjhPQ83883DUqn0lCXJHs1r2H8laJlZIcISSgPoStxnt/ni6ZxEWnISHLpZWXcXUro/35b'
        '3lyM0itkqNAnHu4LvKFxy1w4tmkblsT9zXf2kxfI6g6j6lxLTkvzPuj7b2K+HQEobjjAreHKUXpJBGhaPyGdW4qgJ7MNys9H'
        '45VZ27iZh3vwz5fJrOF2AejesKDewzrw22kRLhHa06f5hGisz4HAA9vwdLQnt8lACLdFF7gnOZ8x+W4WHnDfSE+GJ6Hw1n7c'
        'uU2Miusu4KjfU0GixU3QCfmCJj3LGIRNgu2APu19IASnrHzQS64kBVWtYJhDMe/uXIi8rUYfXigE05wK8FqhgcHVCkz7zCRR'
        'jxglhtIOSJrWgUxAAnF1GsAKbSW6XT2NXLZPg46rOXjwqSbw7WLo2FFxqv5zGNveYoXkjBW1OsKj7tZNUGT4Ciy334Njv97B'
        'CHU1lJBVhuSrR7iv9wvJyOEGnAi0YJPqr6ApIhS1XiuBXpE81EfmQ/OsC7Cz8AJ93rILXZKjQdXECDtYCtifrkKl/ZtFVzrL'
        'ibHbECgEauDTFyJitGACJz73c66OZlTu4Vp0s55NLcK02Yt3xyDDJgvu+RrAyol6TO8rRf2fUznVm+Zc6zVlDEvIAvbEloQn'
        'L2GcBoCTR7GoMMYX/RMjUSyIkqBrstybBxoYH38ddp5SoJ7tn2HHhgGU6nFAPfERDF0QC93OqqygxxS6v86DhV/DWLKqDNvT'
        '58k+riqDAP18tLtZTeoLEmlqZxr1j9zI+vVywaEoCt+nnxPlj8aRK8LZlPenHwlq7sQN3esYv+wCag7Fg92MLTvgvpaL8U7B'
        'K8/zMCnLkahIZcK/KatZc6E7lTWfzWJ1zBgu1aBzFPm0r8EdoprK4aRLIdk+9B2jJWxASqKLCIPM6b3t+4nUvfUQ4BGHz8ZM'
        'sMFBTvDmHyW8KbsFR23lKc/vLFT5F3AeP5C0aYn490YTYKIwjhQp+2Fwngs7oBMHz46cB8PwHfTSmYVMXeDOjmWkY/u2Nn6Q'
        '+iy83/VVsF/lBTe5txZfahyCgcbL+KFgA+R9VKHGI3vw6Z6rEN5siXY6pqhx6C9w/BSDNXFyWFv/npSJxvDRtCG70mFE/3dm'
        'B1Z5/kLuj2SB5b4q0qCfAu2mqnhWKYp8m90B4ovzuKar9aDQo8Q8sryo2N61sP9LI9maIE/YZksmUfADHyb5wLriO7ivSJJZ'
        'z4nGRNl0kD38ECp1j0Owxm1upFCPTqWJQWu3MvWX9wKXIRlWnbQCtE97kfp8Szro+Va08FdDLrrUDBcvHoW6CmX0cXLANeUC'
        'uuXJOPyzMZGU1ksSgwALJnewn1s4tg7FC0JIyL/5lF8yjD7RDPbFKTKDkx+xZBaHSQXu9HcrSfT/0I5D3ySZ7o2XGK29HN7Y'
        '7oIxuSJQWj+KHSlL2bFAPSa1YBKc7s5Dw29uODMpA39t3Iye06aYdnAV7iutwmW77oH0ntWCRuPdNLbfAgVaa9GAWFPTxvvg'
        'esmXPeusQvUfyCV25HEVBxvx+Nk80Fp7B/L7JdhRMz+oSx3C21OPIMmhBeLv9JLjAg+Mqezk2rPmgpLqbLZt8m+oXRoOLabL'
        '2I3dFZBgb4je5pcFg7UJEDzchVJ6QXjOeC07Y2dA3zytIRk8KbbIVgGK58bCMs00NG/RgzlagPw/vMldcyPQTOji/jp3BN28'
        'HMG2sgtvtmjh0E4bXOwnB60Xb3HjG/8TvSqYRX9Xc+VuZSfiG0lXGH2vzYqKH2BtyANkdXJYltuO5Lkh+jrY04vCdkGykTLz'
        '/2DGDJceQ37zDW6vsjOLkMmBWLHznLDxO1Z/v4hTQ7tAWBIK8Y3nICCzCW2V59Dz39XZ3sxyTFE2Qgu3Ili1/gV/fVw8embP'
        'FqnNO0EKGpbCImklop09n4WpGcD0ym4wUTOmiycvQMqhMpJQ7o4ezQXwdpgHOTkosHncBn4K03DNTpEKf9pFmmtjyLf7UbD1'
        '+N8wUCvOSjcQGFY1wIpLHeBUPgbHq73pjsSj8NiwDjYHO7MQx6vE5Ys1e698F8cXHwbnaVsqn/qHIOikPQjuR3KNOEqan6UL'
        'LkvIscdnK7g6b13ma9AAp2LXUElHA0hyH4K0RwoiWCSP6eJW1qcbD8KcV58hUPUZVPVlQ0/WOvB6r8KuvZ/gPPVFqKZsLJos'
        'O07N/5GkPn/Mg4apFeTnIHU2HXKak4k2ZEtLTqBnr4ToZdE4es4bJ7fWT5L+BlMo/C2YXLSTpyWqAbDbbwndpFNM1CXaIWd1'
        'Mk5PacLC6zPoePknSFVaTaxd16Js7FYcCt2MJYP+rDpInmmcb7RKDJdmLyLrcWZnKkpdM2GyJp/wl6tfIVcpG7e0n4QKx/lQ'
        'eMsMNQbLOSLOox+ka7gk00AMkOdVhgf1cjIL22BDpjUw3qjotdcp7m2rMUxmdKNMwkYuYuiLwL47kww81Qf1GQUWOj8eXvAS'
        'OeP5R7EqcIAkLBHiHdME2DW3CNX9u/BSUBq49VxHrfEYTDLNQ3mbGdTIlaIb2mNxv/8mrIvaxImvTEGrBG32SzES+6nDrGtd'
        'ArY+OgXCl5a4qrsSJw4V8DW/ZHEztyL4zsPzsW3BOygZVcYLUymoOLySWokCMdRsQlTbmwu7+qNYjEMInIgKhJy0f4n+L92i'
        '3e0/uF39e1Hp8wg+Tr0D7NtPXMM8O1a3KQSljNRx5hSHebnpkPrkBqTrF+E/Lu/ATmyY269Qwg0Mhgkap/lsVdhRUN6uRttb'
        'MzDD04h5GX0BLU8Vels1Hd1aG1HjRjvRLx6BQdd/MaBbjmZ76TG+wy1oC4oB32kNoGEv0XhFNC6I7MOdxtr03bF5mJuqTN/V'
        'qDHP5/FwouE4fX31Im6NCYWIImXuXqg6uZkeBbdDbdivcT/A40EcPKgYJ9+WnsTqF4SW6T+CvuWZ5JBRDuqFcuToys2sOEYW'
        '05WMWckVWyb98RTaTofD0Q8jOFWuwDwcX2Pe+Dc0f5tFVn2KwA+7f8WHA1a4+q04M528gopfmzGYx2M5RgHo1a1Mqyd1uFtf'
        'y7gXXQdgrLEUn5TGQFxIH6zZEIGGSqbsU3wEMzEohpJgNzhy7jxu/BjG9JWtqcrHYIHFCiH8vcoBwxLvoPqQOVMssyGR0s3A'
        'nTNAsd+vkbzsfrIn+xG26PlUfrooicWTlCuuNGWM3sCz1iVcpeVCpiYMAx/V+fAscx57lKHF/PlR8ErehXsccgAGz43wvWPy'
        'uZ6aUMy/KMaKWDMnX+sH9mOdopQvFmT5cyfYvFwOhq5rsp/JepokHY6DXYHQ5FSPOVoyuLVrVNRR+vjPNfFZoNInoAE117kM'
        'w9/YnM8/wYDZE6ybtQV7JpwxN2gBPaJjRH2fN+Edvgld0zILLa12wkiIPPYaOWOL5yPRaE2vIKP9EkrNLAHjlV5cb4qM4IHO'
        'EWxzm4K2fF+IOLoMY3MVUKWSx3pz4ki1szi1PPEGikrSyfcD9cQszlLElT3Ai3rLgdjyaGNCPV5rShLp5jhCz3VZavD1GWad'
        '6RHs4W0RNfc7oa25Bz2wxp1elVsEnWsCRCd9yrB4UYNI0aewMvyuGBN9rsDYSRe6rHMJPcy/BMGXedRSOMNpdc+huq48draG'
        'R6/Em1LHhzWwMK8L42ZF4IWSQZzZK0dtyhtRGOUrOjhnPpucvZReWmEOkJ0GrooL6LRKucAmR1WAmer0WJ8uFYQ+xpX6g1jz'
        'rY87QU3xZaOh6D+JOpFi2GsUfvTGyHFLZC5xArfljpVRB7dyp7RbIa2tFKziroGj9BgqfT/NyeS9F2UqfsCQL2oQUjWfWWRW'
        '44lNXaKXJAwlD5mw/wM94drs'
    ),
    'How does giving customer support workers an AI assistant affect their productivity?': (
        'eJwFwQcgF2gbAHCyKhpERh1SqUSiEO/zUhGSUl2kIflUqKRkVsqIzHQyIzrJSujo4v8+b1ahI2cnIhHScnUl6cr3+xm5HyGD'
        'ogr0l3or7MqMxeGQbliitZ1suHAewi/8FMjoV+AXFy1Qm/gK6SJCnCmfwv9GtIlkpB7Tz4oA69srwdT7MLVOcUTDulZ07p2J'
        'Y/ozwFnSlS98nQVcbx7NWVgNv6kO43KLPkgDB3z49iqcefcH+dabAu7p3rCmdRc6+lVhovE0GLrdDHG7nmD/cLLAJygfiy1H'
        'iQNrZtede2Gdwl5ecqgE5aVlaXT6E1hh3gt1m5SpzugmJu5ZByltMnxeTQ08/0mokokQNe0+y3IfFuA+hUIMMr0OIkfGMGn5'
        'SjK9bBmCUhkhW+uYW5g5Vd8XRNYmi2PmFjNQlc/Dk+vdaUDTMzBbH85tpV6h1kdXurszjDpvLQfxuhfsVv4yOCMxjDdHH+Mi'
        'NU1wT6gF/zpVVKkW4YcGsvH86llUJQcZsboIV4gsV3LvBBetMYBAO5qj6YddO7wh8+sKfL2xofzRoBtx3BbJBhW8oGpSl9Ym'
        'tKLW1R4mLzeIommdZHipDJ2M1SDvm+bQMeMH2C3laLTkaDH0Kv9C+cRqfle9AUvVZmLjX5wNqwfy2N4oovGunwRQSa4ZpIxy'
        'Z0rx5zwlLvJHMErHdIJQjwjrOloKU0LJMOPPGFyx1Bv8TbX47/7qvDDFlH+ORZJoexMPdP8BQqqBGLbgEj9hF4dWXWMYnV8N'
        'A9e+M18RCfrWI42WnJCkGo8t+PK5GZDTU4czt07ivEZRLNDYD5KH76HpwCwePP6Y1X7cDL7fw9EsV5Nv2JxHjLyUSfOaHLZQ'
        '64Ch4xZpGnbCkl97c5pe3T+XKwvP4AbxlaBVsIpU3TwHtpbxMDNbnukXtWDhqmPkX6dLkGZ/GbYO9YDB5Cxeq6nAm8wbcOX6'
        'CDyOO1Dm9X32WVubfpnIIAoe+pD53LfcbfMCSFQ6h4saxQFq7rOue9r81cYz9Dd9JYgIplR9XJLnxGxDGxdJcsz2IN4/GUo2'
        'pZ7HquIByFpcjc5L01lMRz+qWbqi9NdlbEVJK2yzP8XChAJwQ5E4xL4qAdlLC/mWf+KMlpiXkX3bG8mx/JOsOOwEnVAVpnZC'
        'deicrQA20rKwXFGczG8WwhrXU3CIpRMxyWtMemEG6koizqv4hdbFN7Kam6U43iNGl4/Hc5RYRN6o2YObiiVORehy+VFxSBPR'
        'oW+1CzD0uh2Ji9OAR2NitMyjlXT5baVSC+NhIkEeJRKF4EbLeiqqEEJTnMvQ2MUPR6oMyWLHU+SLWD3rKTmCAjUHarK/gjxt'
        'f0jo6GkyrXgJN5x2i92e+odInLkH6UpR1HHZDSxzecEKbK6Uv96ykTcu/0wcC1fTmvqB8rPSajzLWpdLze1DHq4H5sZa1Df7'
        'Ubm8qSI3tyP8fB1iXpMYE10ph6vVlFG/SZpOVs3jhqp70SPaD591WuHRpaNwxXVsXXS+B5U7L0umr0vFS8d30sjhj2yaoi2X'
        'KdjERhplMeJlInwUu4A1HhKQ3Pc7mKcqsuEgIeoWGY9Fgz2kV0+I3rZwglcD6WxT12yuvaIewrOH8IVfJL1bO41unufNZDT0'
        'CH33NzqGpGC0vzKEBdzEGxI62D4aiyOmipRGm7B74v3rDs8ZgpVLDWhuhD0oeGbCbIUrOEs9kW41s8WeUy9J3s6zEB8cCRG1'
        'YeyihSOeaypiOY2fBK9TN+HpUHGiekwKQtxEyHz5+SC2aAgD5o6hvV0c/uh/j6PwEnV8LQRnzuxk51PvwombIqgTpsZr6rT4'
        '9zIVXhbkhbFq0/mCwRLIcrnG9jpVoF6rKkh1TsdjTuvga5shvRAkyi226YDhY3fMyAoFndwUVE/5wL7MGjAspUcN3/peY6H5'
        '6yBtoIhE6gaxZYVZmKyeQlaEtkN5/Wf4IyqfeIUY8daMceysuUMMIgrRWs4R5U3U4c3FLvCfqUqxQY+6yu6HyvG9uG8B0IPf'
        '5DgsHMK+1k5oeF0Nfcc56EkWQZlNB3aIH4cWZ1MedOEi+2fDHX5WvJT1svlQtuY2iBsdwr7gHwiHI+CbUCrYrVclo1e285Jj'
        'hXDnvQJfXKlBL70JBPgdYc31bjDfrgRHzkWiufpqE9NkRfycVAPfAmvBa3E7hDJOFiTk4dwEa1TvE0LB7mD8fUqd6iuGwOSM'
        'v6Fwcg49emwdk/WKw0vPlLn/10KyKH8vCLZY4O36LLypaQGZ7e/wuqg9a2o+DTtz11Jh2wSyIGon2x1lD4mzU1nwY1PGN+7k'
        'ZY4StH8sBPYknYayyJ08YK0W6Je6c4dwM2r1Xw4Zm67KQ+Yuxu3bg0i81UPc7BOH35gzWWvwxcj033PwviuEJXWfYiMt/mj/'
        'uIg8/d8eorvLhXiuRCbTdQOOHJcBodwEo3UZ1Wz6X8LQrPEMj7a481VNnbitvgKYvSgcbLLEv90/ka42CYqaGyBDpUKQfkoZ'
        'Tj5S57MKJtn9ujbYsncOL8+zhdXRduxe11qw86G4KmUF9/Ish09MiPcJyXKZiSy40uGE+Rnn0HyrtyDimRy/VhlEvsvuAoeN'
        '+rhfJRc3ROVgx/VxzMUpbLhkiUMn5Gn2fAey/edSptNTDoYTyWgip8j7yDgsPmfBY8ziMPnDB3Ywsw8jezthVmsg3Lu6D0VX'
        'evHT/bfYFE6Bv9Re/sFvNv/30W6j2uIO3DFcDtfeNxCLk30spqFWcPnba3jgmwGXn4ahQaUnv/x0lO3B43THh2i0ea/Mt26s'
        'hKRiMfpic7TRnUphUFCLg31aB8DlPydcte0JCBmt4cvnvofBXgIdajGQYbWGZ7SX4VWvLPRMPgLtKSNGsbf64cerNzhpfBM9'
        'fJ1pwLcUVJjjSBKnlwguH+Dkh7E4jx0wQRXjQzCYFAPLeCtUvJOjhhsy4YFpBJnX2I8TDqNYPi4Fjd46fFj0kOCVZxsWN4fg'
        'iavlrOnjPeK85DU6ecnjGq8gblxpgJXtGoJwgz5iVe4IZg6mxO1WM76cI8Gblk1iccszvH/rEdl89hOc0JoPM3t3Y55LAbzU'
        '/YzSqfqYHrUbFmbE4mRZE85ytuPtV6qYzelF8M5xjJn8uhIX+maCz88uQWhADAuO3WXkzlsE2qJ+XLLkOCzuToVXhzzIBXMx'
        '+uisDQZ55jPrtUfQxi1OEGdUim0+b/Fy1BI4e3cXSsyx5rfLh7E+NgimxURgVk0bqpvuYe0PBIKs2XHgfyUdVtQkQmPWRcxN'
        'V+FF//5HbvhHQ6+TN1S0xSOb4KjxbgKC94yy055F4Jq2BAdnpQo0AufzTRfdWa/LfHwS2ggvnmWzv2f0wJar9hzCzPja8XRQ'
        'k9ZFG+lPgsup9vjDSI7+eXE2TdDMxWzvevBin9HS6blgVb0bOiTFo0J+MzysewAL/OJQPCmPjfiq49WWavw4Ug1hFT+g3z0E'
        'BLuXcvOiVCL6mxbX67XDLV4CGJLUpIeCxcjA2UGBbUka/FQS5S7CoSB0QZweKFjLfc4vIsH7h6DneSA1GVLjVTXaRj45Bej6'
        'yYr/2h4BpYtr4eG92XQ0o9MoQeEO+y0zDr4+FkW11b2sWDOEnWPnQObrftow+RzkDnyGCX8zXvrFjbrO6GDChT70Wr4KVcY4'
        '6t5szHqShXn3wTrSKLyDRj0JNzKWegRLomfwactuQWnbbdx5TIOOFs+mlxZZwoJuFVruEglTaRzah0V4h8YznGMtCtke2fBd'
        'vRwbg3RR+24ANvu3oE/wd0HXL8dZ0tIqaPMcgby4v+DiYScqayAJqVrRqLDjKToMbMKpdD8+aM2YolIrVgTO5Ue8QzDTsRrT'
        'wnX5/wG+et+V'
    ),
    'Which study discusses AI use in coursework and exams?': (
        'eJwFwQcgFmgfAHCUzCKrkLyKSDTMvM//qTiq4yTjUkjftUuRijKqS6HMMs5I82SEJEne5/9o0aThklTIjGwq12V8v5++2FrR'
        'lWefganUC3XdakhgiBjsfyTFk3YcAamqFOFho1w0HJ4GWnsOwr1Ud6642wlDHshgpm4Y3tN5QZ7mnyGeeSngdlkAD8KjcExK'
        'iZjKjrIrEqd4x/hp2PFtGq1uLAK9eZr8S2E39JJs3JwlSccfXcCGJiEqdV6DDwq38R8Tdd6g+pB0F5+l1jevYsyZMlQSZJOm'
        'V6uF8edHyJXrklQ2Yj4v2BqEnkFhEKxXCSY+knTdWC84P3PFJK9mWC25Eq+Xt0LFl6eAXSXQ6HMSrH372bUSVd6n74OaR+fh'
        'p/v+6GXrCylZfsRs6Cfu2jSV5mgegrT7a8FpvSzGOgbDugfxNDmniwTMP8GbL3F0Mg+n3SlL6KOZ4nRg7CmZrnFQFPXpKh6o'
        'isd10TcgLfg8y096wozlG3FiYyO+bR+AV5XJaD9bmi6scWHm01Topt/WQHmaH01e5YsFcBXua1qT7Zt9YU3WAK5a8k2U8dgW'
        'LFw64HHNT2YuugR2Q4rcaokn23J7Jt21bQkOhBpR4iHHWdef8NqzGZaEm4LLcAfK+Smht/WQaPDUFlZ3cAgfLlCH1022GJi8'
        'glfGSsBI2QumfkUcW8+Vsp5iCehedBxM9+nDXqkWsLiZAA41TaA6rscD8mpxaLU4P/z9DlgPCHjp9Y3gEKuI/V9Pcl2X27jF'
        'ZCfP1RSDAxX+GH/tPNjqJtHzJpFUFL2UN/Yfh67ppsR0eC0uau1lVyzTYGnHSVKcWI2aMa48t04GLLOrWEiFNdfNkMDTLhYY'
        'utqA28rPZimqCOMO0/j9gWjqaqDAJQWreNlDK7rS9CnYfGwjBq2vIDm1jUhiDHqmxLIG9zbmqjsCX8Zt4KC7PxtZ3coaMyJY'
        's5kZPLuVAo4T9/BNqTLIRD5iw5rSbIb+ZbLJ6DaJuVUm0vf7RjTO2uCgZQTf6yNP/1I9BQW/X6JHXqhxj0McG6OdWG2QON/6'
        '8Qq+XGRO4qUjgQdK8hNuVrD2xjiuJl1C3R+lENGuxkxN3Zlu+m6cWnIWRa374I+tPvig4T42JUbC/hFZrjLKiOd3Q5q4dyZR'
        '98vFp6Hld0prDoPdp6tMvrmBbS96A97v7WDxtwCiM/qStAjaMeFcItX9L4Y9jtFnuz/WsBxpFX5USY7Xjs2lJvV/o1fnZK4m'
        '7sFOBwbBu9L5GNA1CRMdVoA2zKYGas3s/ayFNGh8GNRFfajjuwveJyjRdxUh9LNcBelJsiQl6ebMcjQMbu0FYtz3GMMUbeid'
        'pWpU9s0N2JXygNiNyfIYVXXYr7cY83JTYJXjUTrjiDO22xYQckCbD/9YxdErEaeImdHSOiNkqqvwXvZ0PuF9E4Nl/oIejckw'
        'uuY8qB0T8qFbUnzHnrXcSkIJjsRpCff9KGYHbXKh4/0c7izwwy2/xOAs+2bMW/ENSK2/yCRzD729eRr5xW4RrlR0oO8T3oKt'
        'oy2/VjMbeaIUkOlxWPshDzX7U0nctxzQWfyclL+0ZA7dS/lJo3jyKUWXemRpkMbbTni1fTLuHlPGsxHuPP2QNV0tekOGErsw'
        'nrSANn3NEi0DsMo6nO19EoHi99LQJOQJJrt/g8qxcHQtVMb2aHUqmqpGt6nuFqrN/gE7iy4y/8YC8Igqh9/CCvHcx0J2zdtG'
        '+G9zKW5bG0eyhjhqNB1k2JGDx5f9gjc7k2CWdQ+pEIyz4bxh4cCv0/ge1RSsehuFVcfG2JzQXBx9OcKmhZ+CpO8K0Domye81'
        'reFvqy14t6Qp/hOzjKelrYTxzS9I3U8BL5WNZZUL84hy1goo8p0CQ7FFOEhLwMI+WmTe/5g9LdHF/Gdy1Nj6EZM+t5HZHC8n'
        '9TdmWv2zQQMCTxBQ+0WTB0ctEN2VvwUt922o4ecT4Lg+G/8YasCMg/UQeFEaneMioPpuAuxIuAzlv7vSzdpBoFOwjhV8fCnc'
        'tFCGzriozANteoRbv26FkAOTaIJzN0TVC+izlzpwWnkzUS/35G5p8tRpjQMf79TBQm8XOF2XCrU9bSTnggy6f/0Ts99PgTqP'
        'EdYT8QifOyiD9vJPOD1zLv3Ru5BGpXeSp/MbIDL6FM4w/InLLpsvbz5bBpMOZILpnkRQCW2A0sV+oDFJh2c3xGFrSzrL330G'
        'F26ypscLr4LcNi0yc0UTDDq5c5ln2fhy+yBuHrZBlf5F0NuhyQc8R9i/gjYy7cCQ0M0iUVj3hxY9Cio047QYxYoXJK4lHJ6X'
        'D6O52DaY2y7PlUNHoG1xFUmXUidrT15Du+CdZP64PU8O6EeJzFgWedSe5x8ZIHadD4QKDv1489ktPBK7EvS+/Yn5GcUANZLk'
        'jsFxlPpPWzRjUwF8brFDu0JF8nbGTHjhowKKe0UQfkibHbgbj2cyvhB9vTlYHm/B4wf8MalfgvrazIKg678RiT03mHvwEPwX'
        'V0byHaeSxHxNq8eVKjxHm0LA/7pJwW+vsN8sCJquF2GTRBQaDzpgkCgKS81k6U33MjRmn7GzIw/CtimjoaMHTz6TKFSvl+Tf'
        'M46iscdZgDQN9OvVRbknFXhm+ALO/pGM48uT0Of+bFo2ocV2nFuJ5XMV6aZYF3wq7oCVYkMw76svb73IyHJRDul5LRI+268M'
        'T9cogdcpD2yVUeJSSwqZrrg0zN5pwP/W/4473dyxJroGr935C2L7y8DgdCSetfoAhqGP2WQTH2juEiN9vXH8spQZVKZNoj8M'
        'E/FWgxI39PsK34yrofm1gL+x53j93TJYk2EJM63DRSEgTl3nSXOjWZpUL1cWLuQagUW1GO/2zGIp+8LxYUA8HHIPYLUG+rR0'
        'fAU+t66Dey3r6ET6KPP3Ok6CnbWxKVwEMf+Vkx33j+PwwUnEuU+cCs01IOyDFVQnq9DrV7XIUdGf+CInGOfPloSLWyfzgRUx'
        'zMIzCE9rvEN8no2qYosgOuM1Ts+KxDN3nbkwrw5JlppoxaP3rH9WLKHKs4Rt1QJc3FeMelvaMPJWHBqSMFxTokY3jNqjlrIU'
        'OqktFEa8UsFFByZDakg+MHyFCm4u/M78w/xSrwAtcx7AJsc8zI5ZwlNb5tHakGjm5FkEHW6GmGm3C419fuV3Fl2A5Dn3SP5b'
        'BTQJ8AIqlIZxzybcPpCBle0KvDt0nNktFXK5HIr9dQ54ssKf5520woIBWWonSMJBM0P8Wy0EtPRy2WedbEg8H0YmwrXgodYo'
        'pl9R5XPiUkiyrj2MTpmCvuKt7IOPAnCvZXAs5ieuF88B364hdmVKGtFYoMztD0+g9rZUCPz0jkWaXIBqJynapnkTTV0C+OBw'
        'Lpz6uAC/ailipusx3Fn+GJYZxUN9Xg8bnWND7V+5WuUJ9jGT+gmmcKMEf5zsEabuD4JYgQDnbiCw9eUK1vrHIUzwLoITrcVg'
        'dDgSHIPS2bkCSxLfv5jHHrFmZX/3waU9DGp2RpKzcsdZQVcBOstlol9tO2gkdMIC3oZHZQ3ZIyl3sH9lS/fMlOEKvASW2bqh'
        'lLyA3wv9nR5VP0wX6GeQVUFTwWyHMXOdoQTe4tasWMkHd7iMCoWZerRSoQqCt4vR1kNFhPe8RYkvxZC2PREHNTfSussGVIUn'
        '0LC+GDa6/AJqCZKxYok9HTObzBuVlwDZT/hjqy00624B8bIzhpiOhTTcbzGeWHsZIvQFdN0IULtPU3lZz3q+XnOCyXjvE/3a'
        'uYrUH1mN51LTUHtqH/77+xNmL3+VXP83HN5bFcNo6RcQD8yGQ//4w4aP0nx/4QNWMHU7buj14psNJ5FYZzFetXEYS/d540hR'
        'IzOovo3/B5Go5Xw='
    ),
}
cached = {
    text: struct.unpack("<768f", zlib.decompress(base64.b64decode(packed)))
    for text, packed in packed_vectors.items()
}

In [ ]:
# Turn this on to encode edited notes or questions using the model above.
recompute = False
texts = [text for passages in chunks.values() for text in passages] + [productivity_question, search_question]
if recompute:
    vectors = dict(zip(texts, encode_texts(texts)))
else:
    if encoder != cached_encoder or any(text not in cached for text in texts):
        raise ValueError("The model or text changed. Set recompute=True to create compatible vectors.")
    vectors = {text: cached[text] for text in texts}

first_chunk = example_chunks[0]
print("Dimensions:", len(vectors[first_chunk]))
print("First four coordinates:", [round(x, 3) for x in vectors[first_chunk][:4]])

In [ ]:
import os
import sqlite3
import struct
from pathlib import Path
from tempfile import TemporaryDirectory

# Default: a fresh, disposable database. An explicit file can persist across weekly runs.
temporary = TemporaryDirectory()
database_path = os.environ.get("PAPER_LIBRARY_DB") or str(Path(temporary.name) / "library.sqlite")
library = sqlite3.connect(database_path)
library.execute("""CREATE TABLE IF NOT EXISTS specter2_passages (
    paper_id TEXT, version TEXT, passage_number INTEGER, title TEXT, url TEXT,
    passage TEXT, embedding BLOB, encoder TEXT, PRIMARY KEY (paper_id, version, passage_number)
)""")

def index_paper(record):
    key = record["id"], record["version"]
    rows = []
    for number, passage in enumerate(chunks[key], start=1):
        vector = vectors[passage]
        rows.append((*key, number, record["title"], record["url"], passage,
                     struct.pack("<768f", *vector), encoder))
    # Replace the whole version atomically, including chunks removed in a correction.
    library.execute("DELETE FROM specter2_passages WHERE paper_id=? AND version=?", key)
    library.executemany("INSERT INTO specter2_passages VALUES (?, ?, ?, ?, ?, ?, ?, ?)", rows)

with library:
    for seed in records[:2]:
        index_paper(seed)
print("Starting papers:", len(records[:2]))

### See what the graph does

The vector index works without a graph. The research tracker also built a graph with papers and concepts as nodes. When screening new work (step 5), it searched ten library passages and kept up to twenty frequent concepts attached to the papers it found. The example below inspects six selected edges from the archived graph, with anonymized paper identities. A researcher can use the graph to find shared vocabulary. A link means the paper discusses the concept.

In [ ]:
from collections import Counter

edges = {
    "Paper A": ["Human Capital", "Labor Markets", "Wages"],
    "Paper B": ["Labor Markets", "Wages", "Automation"],
}
# Illustration: suppose both papers appear among the retrieved passages.
retrieved_papers = ["Paper A", "Paper B"]
concept_counts = Counter(label for paper in retrieved_papers for label in edges[paper])
print(concept_counts.most_common())

In [ ]:
# Draw the six paper–concept edges above. Only anonymized paper labels enter the figure.
from html import escape

def show_concepts(edges):
    from IPython.display import SVG, display
    concepts = sorted({label for labels in edges.values() for label in labels})
    height = 45 * max(len(concepts), len(edges)) + 22
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 520 {height}" role="img"><title>Concepts discussed by the example papers</title>']
    positions = {label: 30 + 45 * i for i, label in enumerate(concepts)}
    for i, (paper, labels) in enumerate(edges.items()):
        y = (i + 0.5) * height / len(edges)
        parts.append(f'<text x="8" y="{y+5}" fill="#30463a" font-size="17">{escape(paper)}</text>')
        for label in labels:
            parts.append(f'<path d="M92 {y} C160 {y} 195 {positions[label]} 275 {positions[label]}" fill="none" stroke="#699184" stroke-width="2"/>')
        parts.append(f'<circle cx="92" cy="{y}" r="4" fill="#346357"/>')
    for label, y in positions.items():
        parts.append(f'<circle cx="275" cy="{y}" r="4" fill="#346357"/><text x="288" y="{y+5}" fill="#30463a" font-size="17">{escape(label)}</text>')
    display(SVG("".join(parts) + "</svg>"))

try:
    show_concepts(edges)
except ImportError:
    print(edges)

## 4. Retrieve the nearest passages

To *encode* a question, we give it the same kind of coordinate as the passages. Think of each coordinate as an arrow from the origin. Cosine similarity compares the direction of two arrows. The closer their directions, the closer the score is to 1. With these unit-length arrows, the cosine is their dot product.

Ask about productivity in customer support. Rank the two starting papers by the strongest passage in each and read what they say. Even a high similarity can come from two passages sharing vocabulary. It is a cue to read the passage, and the number has no probability interpretation.

In [ ]:
from math import fsum

def search(query, paper_ids=None):
    question_vector = vectors[query]
    best = {}
    rows = library.execute("""SELECT paper_id, version, title, url, passage_number, passage, embedding
        FROM specter2_passages WHERE encoder=? ORDER BY paper_id, version, passage_number""", (encoder,))
    for paper_id, version, title, url, number, passage, blob in rows:
        if paper_ids is not None and paper_id not in paper_ids:
            continue
        vector = struct.unpack("<768f", blob)
        similarity = fsum(x * y for x, y in zip(question_vector, vector))
        if paper_id not in best or similarity > best[paper_id]["cosine"]:
            best[paper_id] = dict(paper_id=paper_id, version=version, title=title, url=url,
                                  chunk_number=number, snippet=passage, cosine=similarity)
    return sorted(best.values(), key=lambda row: (-row["cosine"], row["paper_id"]))

starting_ids = {r["id"] for r in records[:2]}
worked = search(productivity_question, starting_ids)
print(productivity_question)
for hit in worked[:2]:
    print(f"\n{hit['cosine']:.3f}  {hit['title']} (chunk {hit['chunk_number']})")
    print(hit["snippet"])
    print(hit["url"])

before = search(search_question, starting_ids)

In [ ]:
# Optional: run the same search in Chroma (tested with chromadb 1.3.5).
run_chroma = False
if run_chroma:
    import chromadb

    client = chromadb.EphemeralClient()
    from uuid import uuid4
    collection_name = f"public-paper-tracker-{uuid4().hex}"
    collection = client.create_collection(collection_name, metadata={"hnsw:space": "cosine"},
                                                  embedding_function=None)
    for record in records[:2]:
        key = record["id"], record["version"]
        passages = chunks[key]
        ids = [f"{key[0]}|{key[1]}|{i}" for i in range(1, len(passages) + 1)]
        collection.upsert(ids=ids, embeddings=[list(vectors[p]) for p in passages], documents=passages,
                          metadatas=[{"paper_id": key[0], "chunk": i} for i in range(1, len(passages) + 1)])
    hits = collection.query(query_embeddings=[list(vectors[productivity_question])],
                            n_results=collection.count(), include=["metadatas", "distances"])
    closest = {}
    for meta, distance in zip(hits["metadatas"][0], hits["distances"][0]):
        closest.setdefault(meta["paper_id"], 1 - distance)
    print([(pid, round(score, 3)) for pid, score in closest.items()])
    assert list(closest) == [row["paper_id"] for row in worked]
    assert all(abs(closest[row["paper_id"]] - row["cosine"]) < 1e-5 for row in worked)
    client.delete_collection(collection_name)

## 5. Screen new papers

The archived RSS batch contains 628 announcements from arXiv (565), CEPR (50) and NBER (13). Its February 2026 report saved the top ten. The following records replay their published titles and recorded score components. They belong to the historical research index; the teaching index is a smaller, separate collection.

The article displays three feed announcements dated 4 December 2025. That is the feed announcement date, distinct from the report date.

In [ ]:
report = [
    {
        "rank": 1,
        "title": "Better World Models Can Lead to Better Post-Training Performance",
        "url": "https://arxiv.org/abs/2512.03400",
        "semantic": 0.066,
        "graph": 0.1,
        "rerank": 1.0,
        "recorded": 0.443
    },
    {
        "rank": 2,
        "title": "DP20848 Strategic Avoidance and the Welfare Impacts of U.S. Solar Panel Tariffs",
        "url": "https://cepr.org/publications/dp20848",
        "semantic": 0.087,
        "graph": 0.1,
        "rerank": 0.828,
        "recorded": 0.385
    },
    {
        "rank": 3,
        "title": "Semantic Trading: Agentic AI for Clustering and Relationship Discovery in Prediction Markets",
        "url": "https://arxiv.org/abs/2512.02436",
        "semantic": 0.075,
        "graph": 0.1,
        "rerank": 0.716,
        "recorded": 0.334
    },
    {
        "rank": 4,
        "title": "DP20867 Oligopolistic Information Markets",
        "url": "https://cepr.org/publications/dp20867",
        "semantic": 0.081,
        "graph": 0.1,
        "rerank": 0.608,
        "recorded": 0.294
    },
    {
        "rank": 5,
        "title": "Pricing AI Model Accuracy",
        "url": "https://arxiv.org/abs/2504.13375",
        "semantic": 0.113,
        "graph": 0.1,
        "rerank": 0.451,
        "recorded": 0.247
    },
    {
        "rank": 6,
        "title": "Enhancing Job Matching: Occupation, Skill and Qualification Linking with the ESCO and EQF taxonomies",
        "url": "https://arxiv.org/abs/2512.03195",
        "semantic": 0.074,
        "graph": 0.1,
        "rerank": 0.449,
        "recorded": 0.227
    },
    {
        "rank": 7,
        "title": "Equalizer or amplifier? How AI may reshape human cognitive differences",
        "url": "https://arxiv.org/abs/2512.03902",
        "semantic": 0.159,
        "graph": 0.4,
        "rerank": 0.264,
        "recorded": 0.225
    },
    {
        "rank": 8,
        "title": "Eval Factsheets: A Structured Framework for Documenting AI Evaluations",
        "url": "https://arxiv.org/abs/2512.04062",
        "semantic": 0.06,
        "graph": 0.1,
        "rerank": 0.461,
        "recorded": 0.224
    },
    {
        "rank": 9,
        "title": "AI Text Detectors and the Misclassification of Slightly Polished Arabic Text",
        "url": "https://arxiv.org/abs/2511.16690",
        "semantic": 0.065,
        "graph": 0.2,
        "rerank": 0.423,
        "recorded": 0.222
    },
    {
        "rank": 10,
        "title": "The Economic Legacy of Racial Trauma in the American South",
        "url": "https://www.nber.org/papers/w34523",
        "semantic": 0.084,
        "graph": 0.0,
        "rerank": 0.39,
        "recorded": 0.198
    }
]
print("Papers in the saved shortlist:", len(report))

### Calculate the three signals

**Semantic.** Embed the incoming title and abstract, find the three closest stored passages, convert each distance to 1/(1+d) and average them.

**Graph.** Count how many query-relevant concept labels occur as case-insensitive substrings of the incoming title and abstract, then divide by ten and cap at one. Short labels such as “AI” can also match inside unrelated words. Inspect the matched labels.

**Reranker.** Score every candidate against the original research question. When enough concepts are available, score up to two concept-expanded versions too. Prefer ColBERT with a cross-encoder fallback. Min–max scale the scores within each query pass, then keep each paper’s best score. The old code records zero if this component is unavailable, so an operator should log model availability alongside each report.

The following inputs are small invented numbers to illustrate the calculation. The archived report preserved component scores, not raw embedding distances or raw reranker outputs.

In [ ]:
from statistics import mean

def semantic_score(distances):
    return mean(1 / (1 + d) for d in distances) if distances else 0.0

def graph_score(text, concepts):
    matched = [label for label in concepts if label.casefold() in text.casefold()]
    return matched, min(len(matched) / 10, 1.0)

def rescale(scores):
    low, high = min(scores), max(scores)
    if high == low:
        return [0.0] * len(scores)
    return [(x - low) / (high - low) for x in scores]

print("Semantic example:", round(semantic_score([0.4, 0.6, 1.0]), 3))
print("Graph example:", graph_score("Automation, wages and labor markets", concept_counts))
# Rows are query variants. Columns are the same three candidates throughout.
raw_passes = [[1.0, 2.0, 4.0], [3.0, 2.0, 2.0], [2.0, 2.0, 2.0]]
print("Best reranker value per candidate:", list(map(max, zip(*(rescale(row) for row in raw_passes)))))

### Replay the historical report

The next cell uses the stored, rounded inputs from the original top ten. Weights were 50% semantic, 10% graph and 40% reranker. The original runner combined components at full precision before rounding, so reconstruction from displayed components can differ by about one thousandth.

In [ ]:
components = ("semantic", "graph", "rerank")
original_weights = {"semantic": 0.5, "graph": 0.1, "rerank": 0.4}

def score(paper, weights):
    if set(weights) != set(components) or min(weights.values()) < 0 or abs(sum(weights.values()) - 1) > 1e-9:
        raise ValueError("Use three nonnegative weights summing to one.")
    return sum(weights[key] * paper[key] for key in components)

for paper in report:
    rebuilt = score(paper, original_weights)
    assert abs(rebuilt - paper["recorded"]) <= 0.0011
    print(f"{paper['rank']:2d}. {paper['title']} ({paper['recorded']:.3f}, reconstructed {rebuilt:.3f})")

### Change the weights

The first-ranked paper studies model training and received a reranker value of 1. The seventh examines differences in human performance, matches four active concept labels and received a higher semantic score. Inspect which part of the ranking changes when the library gets more weight. This experiment reorders only the archived top ten. Papers excluded from that set could enter the top ten if the entire original batch were rescored.

In [ ]:
alternative = {"semantic": 0.60, "graph": 0.30, "rerank": 0.10}
for new_rank, paper in enumerate(sorted(report, key=lambda p: -score(p, alternative)), 1):
    print(f"{new_rank:2d}. {paper['title']} | original {paper['rank']} | score {score(paper, alternative):.3f}")

## 6. Review and expand

Suppose our next question is about AI in coursework and exams. Neither starting paper studies assessments. In the historical shortlist, the paper on cognitive differences discusses that question. Mark it *keep*, add its short teaching note to the database and search again. Comparing the first-ranked passage before and after admission shows how a reading decision changes the evidence we can retrieve.

**Try it twice.** On the first run, look for *Equalizer or amplifier?* as the closest paper after review. Then change its decision in the next cell from `keep` to `pass`, restart the kernel and choose **Run All**. With the default temporary library, the customer-support paper will remain the closest match. Read its passage and decide whether it answers the question about exams.

In [ ]:
decisions = {
    "https://arxiv.org/abs/2512.03400": ("pass", "The paper studies model training."),
    "https://arxiv.org/abs/2512.03902": ("keep", "Extends the library to human performance and assessment."),
}
notes_for_review = {"https://arxiv.org/abs/2512.03902": records[2]}
library.execute("""CREATE TABLE IF NOT EXISTS reviews
    (url TEXT PRIMARY KEY, action TEXT NOT NULL, reason TEXT NOT NULL)""")

def apply_review(shortlist, decisions, prepared):
    for candidate in shortlist:
        url = candidate["url"]
        if url not in decisions:
            continue
        action, reason = decisions[url]
        if action not in ("keep", "pass", "defer"):
            raise ValueError("Review actions are keep, pass, or defer.")
        with library:
            library.execute("""INSERT INTO reviews VALUES (?, ?, ?)
                ON CONFLICT(url) DO UPDATE SET action=excluded.action, reason=excluded.reason""",
                (url, action, reason))
            if action == "keep" and url in prepared:
                index_paper(prepared[url])

apply_review(report, decisions, notes_for_review)
after = search(search_question)
count = library.execute("SELECT COUNT(*) FROM specter2_passages").fetchone()[0]
apply_review(report, decisions, notes_for_review)
assert library.execute("SELECT COUNT(*) FROM specter2_passages").fetchone()[0] == count
print(search_question)
for stage, ranked in [("Starting collection", before), ("After review", after)]:
    hit = ranked[0]
    print(f"{stage}: {hit['title']} ({hit['cosine']:.3f})")
    print(hit["snippet"])
    print(hit["url"])
print("Passages after a second run:", count)
library.close()
temporary.cleanup()

### How would a weekly tracker work?

A live tracker would need a schedule, current feeds and a persistent library. Each week it could collect new announcements, rank them against the library and save a shortlist for a researcher. On the next run it would add passages from the approved papers. Here the saved shortlist lets you repeat the same reading decision at any time.

For an experiment that keeps its results, set the optional `PAPER_LIBRARY_DB` environment variable to a dedicated teaching database before starting the kernel. Step 6 then searches all papers previously saved there with this encoder. Leave it unset for the two-run exercise above.